# Import packages and data loading

In [1]:
import time
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src import (
    load_instance,
    build_model,
    solve_model,
    solve_and_summarize_all,
    extract_assignment,
    print_summary,
    get_model_stats,
    display_solution,
)

import json
import pandas as pd

in this section we work with models that use Hungarian policy to solve problems with possible ties. bacause of this, we don't need preprocessed datasets and the simple datasets are enough.

In [2]:
data_small = json.load(open(r'../data/generated/instance_small.json', 'r', encoding='utf-8'))
print("Loaded small instance:\n   n={}, m={}".format(data_small['n'], data_small['m']))

data_medium = json.load(open(r'../data/generated/instance_medium.json', 'r', encoding='utf-8'))
print("Loaded medium instance:\n   n={}, m={}".format(data_medium['n'], data_medium['m']))

Loaded small instance:
   n=10, m=5
Loaded medium instance:
   n=50, m=20


# Solving Models

first we run the Pyomo models for Chilean policy:
- `SO-C-NW-CUT`
- `SO-C-NW-BIN-CUT`

these models makes it possible for ties to violate quota constraint to accept applicants with same score.

In [3]:
section_four_chilean_formulations = ["SO-C-NW-CUT", "SO-C-NW-BIN-CUT"]

results_small = solve_and_summarize_all(data_small, solver_name='cplex', tee=False, formulations=section_four_chilean_formulations)


print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))


SUMMARY TABLE
    Formulation  Status Model Obj Rank Obj  Matched Avg Rank
    SO-C-NW-CUT OPTIMAL     31.00        9        8     1.12
SO-C-NW-BIN-CUT OPTIMAL     31.00        9        8     1.12


# Policy comparison
in this section 